# One large MLP versus the 5-seed ensemble

Repeated stratified 5-fold CV (3 repeats, seeds 42-46, epochs 100-600) on train + val for two single HOG + HSV + Lab MLPs with the parameter budget of five 256-64-32 MLPs (3,646,590): wide 1200-256-64 (3,654,934) and deep 1056-512-256-128-64 (3,646,374). Folds and seeds match the 11 Nov CV run, so its out-of-fold scores can be compared pairwise. The test split is not read. No pretrained weights.


In [ ]:
import base64, hashlib, io, json, os, subprocess, sys, time, zipfile
from pathlib import Path
import torch

SOURCE_SHA256 = "3a9e88bb29dee758c750de5af7b5df6fea8d5f433124eb1d220f6705a3b7138b"
SOURCE_BASE_REVISION = "c846958a90c5e212b4f523c78f45af0f7ab63196"
payload = base64.b64decode("...")
assert hashlib.sha256(payload).hexdigest() == SOURCE_SHA256
root = Path("/tmp/wastelens-cv-snapshot")
root.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(root)
run_root = Path("/kaggle/working/wastelens-single-vs-ensemble")
run_root.mkdir(exist_ok=False)
(run_root / "source-snapshot.zip").write_bytes(payload)
(run_root / "source-snapshot.json").write_text(json.dumps({
    "sha256": SOURCE_SHA256, "base_revision": SOURCE_BASE_REVISION}, indent=2), encoding="utf-8")
assert torch.cuda.is_available(), "Requires a Kaggle GPU (T4 x2)"
devices = [(torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
           for i in range(torch.cuda.device_count())]
print(torch.__version__, devices, "CPUs:", os.cpu_count(), flush=True)
torch.ones(2, device="cuda").sum().item()
subprocess.run([sys.executable, str(root / "scripts/prepare_trashnet.py")], cwd=root, check=True)
env = dict(os.environ, PYTHONPATH=str(root / "src"), CUBLAS_WORKSPACE_CONFIG=":4096:8")


In [ ]:
# CV only: the test split is not read by this command.
log_path = run_root / "cv.log"
with log_path.open("w", encoding="utf-8") as log:
    worker = subprocess.Popen(
        [sys.executable, "-u", "-m", "wastelens.cv", "cv", "--device", "cuda", "--workers", "4",
         "--threads", "1", "--cache", str(run_root / "cache"), "--output", str(run_root / "cv"),
         *['--configs', 'extra_lab_wide', 'extra_lab_deep']],
        cwd=root, env=env, stdout=log, stderr=subprocess.STDOUT)
    last = None
    while worker.poll() is None:
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        if lines and lines[-1] != last:
            print(lines[-1], flush=True)
            last = lines[-1]
        time.sleep(30)
print(log_path.read_text(encoding="utf-8", errors="replace")[-4000:])
if worker.returncode:
    raise RuntimeError(f"CV failed with exit code {worker.returncode}")


In [ ]:
# Descriptor caches are reproducible from the snapshot; keep the artifact small.
subprocess.run(["rm", "-rf", str(run_root / "cache")], check=True)
